# SRQ-FLY Priority 3: direct-quantization control
Train-only CIFAR-100 ablation. This notebook never materializes `test.pt`. Run every cell in order on a T4 GPU and return the final ZIP for audit.

In [ ]:
# Edit repository/path values only. Do not edit protocol, repair, seed, or gates.
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_DIR = '/content/srq_priority1_cifar_features'
WTA_CACHE_DIR = '/content/srq_priority1_wta_10000'
OUTPUT_DIR = '/content/srq_priority3_direct_control'
BATCH_SIZE = 128
NUM_WORKERS = 2

In [ ]:
# Fresh clone, dependencies, GPU check, and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG='configs/srq_fly_priority3_direct_control_cifar100_train_only.json'
RUNNER='tools/srq_fly_priority3_direct_control.py'
EXPECTED={
 CONFIG:'909fbd4da753018c3e8b3e8dcd8ea33e78dc5c5f6f32880035875d2b2b51e4dd',
 RUNNER:'ca68754d50fa0e5de467064d519230adb12f5fb28440d85144a399b733ef3b80',
 'methods/srq_fly_optimized/learner.py':'40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae',
 'methods/srq_fly_optimized/storage.py':'9d288a3661985da657371e8581f406825d4a8d5e6e0c63381aacda8484490986',
 'methods/srq_fly_optimized/direct_control.py':'84a680eadaac5dbd80091a47dfb66cb0dadddc9524356b6ea20b36405e58dbf0',
 'tools/srq_fly_priority1_ablation.py':'b65ce01bfc2e2f9f07a61ecd056637156b504626c74c45b6aa3a7c8162e22136',
 'tools/srq_fly_d0.py':'60566c92512a97ae49d981b746f6c91c477b3d8dade3194ec6fa4b51f9c3619a'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
print('GPU:',torch.cuda.get_device_name(0))
print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('SOURCE HASH GATE: PASS')

In [ ]:
# Synthetic proof-contract, checkpoint, state, and regression gates.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_fly_priority3_direct_control.py','tests/test_srq_fly_priority1_ablation.py','tests/test_srq_fly_optimized.py']
completed=subprocess.run(command)
assert completed.returncode==0,'Correctness gate failed; return the complete pytest output.'
print('SRQ-FLY PRIORITY-3 CORRECTNESS GATE: PASS')

In [ ]:
# Download the exact frozen checkpoint and processed CIFAR-100 artifact.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('checkpoint:',CHECKPOINT_PATH)
print('CIFAR-100:',CIFAR_ROOT)

In [ ]:
# Reuse or extract TRAIN features only. The held-out feature cache must stay absent.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_priority3','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Five isolated paired controls. Safe to rerun: source/config-matched rows resume.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--code-cache-dir',WTA_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda']
print('PRIORITY-3 START: Exact, naive direct INT8, repaired direct INT8, FP16 sqrt, SRQ P2B.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'priority3_results.json'
assert completed.returncode==0 and result_path.is_file(),'Priority-3 runner failed; return the complete output without changing config.'
summary=json.loads(result_path.read_text())
assert summary['status']=='COMPLETE_REVIEW_PRIORITY3',summary['methodological_gates']
assert summary['uses_test_set'] is False
print('DECISION:',summary['status'])
print('INTERPRETATION:',summary['interpretation'])
print(json.dumps(summary['accuracy_deltas'],indent=2))

In [ ]:
# Compact audit tables and plots. All accuracy values are train-validation only.
import pandas as pd, matplotlib.pyplot as plt, numpy as np
rows=[]
for row in summary['results']:
    rows.append({'method':row['method'],'status':row['status'],'validation_AIA':row.get('validation_average_accuracy'),'state_MiB':row['persistent_state_bytes']/2**20,'peak_allocated_GiB':None if row['peak_cuda_allocated_bytes'] is None else row['peak_cuda_allocated_bytes']/2**30,'update_seconds':row['total_update_seconds'],'max_solver_residual':row.get('maximum_solver_relative_residual')})
table=pd.DataFrame(rows)
display(table)
complete=table[table.status=='complete'].copy()
fig,axes=plt.subplots(1,3,figsize=(17,4.5))
axes[0].bar(complete.method,complete.validation_AIA); axes[0].set_ylabel('Train-validation AIA (%)')
axes[1].bar(complete.method,complete.state_MiB); axes[1].set_ylabel('Persistent tensor state (MiB)')
axes[2].bar(complete.method,complete.peak_allocated_GiB); axes[2].set_ylabel('Peak CUDA allocated (GiB)')
for axis in axes: axis.tick_params(axis='x',rotation=65)
fig.tight_layout(); FIGURE=Path('/content/srq_fly_priority3_summary.png'); fig.savefig(FIGURE,dpi=180,bbox_inches='tight'); plt.show()
repair=next(row for row in summary['results'] if row['method']=='direct_int8_gram_weyl_repair')
repair_rows=pd.DataFrame(repair['task_diagnostics'])
fig,axis=plt.subplots(figsize=(7,4))
axis.plot(repair_rows.task,repair_rows.diagonal_loading/1e6,marker='o')
axis.set(xlabel='Task',ylabel='Additional loading / base ridge',title='Certified direct-INT8 repair strength')
axis.grid(alpha=.25); REPAIR_FIGURE=Path('/content/srq_fly_priority3_repair.png'); fig.tight_layout(); fig.savefig(REPAIR_FIGURE,dpi=180,bbox_inches='tight'); plt.show()
print('Repair observations:',json.dumps(summary['repair_observations'],indent=2))

In [ ]:
# Export evidence only. Per-sample feature/WTA caches are deliberately excluded.
bundle=Path('/content/srq_fly_priority3_direct_control_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree(OUTPUT_DIR,bundle/'results')
shutil.copy2(CONFIG,bundle/'locked_config.json')
shutil.copy2('docs/research/SRQ_FLY_PRIORITY3_DIRECT_CONTROL_PROTOCOL.md',bundle/'protocol.md')
shutil.copy2(FIGURE,bundle/FIGURE.name); shutil.copy2(REPAIR_FIGURE,bundle/REPAIR_FIGURE.name)
manifest={'artifact':'srq_fly_priority3_direct_control_train_only','uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_hashes':EXPECTED,'summary_sha256':sha(Path(OUTPUT_DIR)/'priority3_results.json')}
(bundle/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
from google.colab import files
files.download(archive)